# Web Search Tool — Notes

## Core Concept

A **server-side tool**: Anthropic provides both the schema AND the execution. You just flip a switch.

## My Summary (in plain words)

Sometimes we want Claude to search the web for information. Since websites are publicly accessible, **Anthropic's servers can perform the search on our behalf**. So we only need to drop a stub into the API call — **no need to define a schema, no need to write an execution function**. Anthropic's servers run the search automatically, feed the results to Claude, and Claude generates an answer with citations to return to us.

The parts we control are mainly `max_uses` (cap on how many searches per request) and the optional `allowed_domains` (restrict searches to trusted sites).

**Key contrast with Text Editor Tool:**

- **Text Editor**: Anthropic provides the schema, the developer writes the execution → *semi-managed*
- **Web Search**: Anthropic provides the schema AND handles the execution → *fully managed*

The underlying principle: whether the operation's target is something Anthropic can directly access. Local files live on your computer — Anthropic can't reach them. Websites live on the public internet — Anthropic can fetch them whenever needed.

## How It Differs From Text Editor Tool

| Tool | Schema | Execution |
|---|---|---|
| Custom tool | You | You |
| Text Editor | Anthropic | **You** |
| Web Search | Anthropic | **Anthropic** |

> With Web Search, you write **zero implementation code**. Anthropic's servers do the searching and return results inside the response.

## Setup

```python
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5   # cap on follow-up searches per request
}
```

Drop it into `tools=[...]` and you're done. Claude decides when to search, what to search, and how many times (up to `max_uses`).

## Restricting to Trusted Domains

```python
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}
```

Useful for medical, legal, academic — anywhere you want authoritative sources only.

## Response Structure

When Claude uses web search, the response contains multiple block types:

- **text** — Claude's answer
- **server_tool_use** — the exact query Claude ran (transparency)
- **web_search_tool_result** — list of search hits
- **web_search_result** — individual result (title + URL)
- **citation** — which sentence came from which source

This structure is designed for UI rendering: show the answer in the main area, sources in a sidebar, and inline citations for traceability (think Perplexity-style answers).

## When to Use

Good fit:

- Current events, recent developments
- Specialized info outside Claude's training data
- Fact-checking, finding authoritative sources
- Research tasks needing up-to-date data

Not a fit:

- General knowledge Claude already has
- Coding, writing, reasoning tasks
- Info already provided in your prompt

## Prerequisite

Org admin must enable Web Search in the Anthropic console first:
https://console.anthropic.com/settings/privacy

(Off by default — search sends user queries to external engines, so it's an explicit opt-in.)

## One-Line Takeaway

The easiest tool to enable: a few lines of config and Claude handles the entire search-read-cite-answer loop server-side.

---

**Interview / work angle**: Web search is a **server tool** — unlike text editor, the developer writes no implementation. Anthropic runs the search and returns results with citations as structured blocks in the response.